# 02 — Fetch Daily OHLCV

Notebook này thực hiện thu thập dữ liệu giá OHLCV hàng ngày cho danh sách cổ phiếu cần phân tích (universe/watchlist) từ SSI FCData API. Dữ liệu thô sau đó được chuẩn hóa, ghép nối và lưu trữ tập trung phục vụ toàn bộ các bước nghiên cứu hạ nguồn.

## Trách nhiệm chính:
1. Kết nối SSI FCData client.
2. Xác nhận danh sách mã (universe/watchlist) và khoảng thời gian lấy dữ liệu.
3. Fetch dữ liệu lịch sử giá cho các mã cổ phiếu và lưu dạng file `.parquet` riêng biệt tại `data/raw/daily_ohlcv/`.
4. Hợp nhất toàn bộ dữ liệu thô đã làm sạch và lưu trữ tại `data/processed/clean_daily_ohlcv.parquet`.


In [3]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path
def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")

project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [4]:
# Cell: Import Fetch Modules
from datetime import date, timedelta
from pathlib import Path
import re
import unicodedata

import pandas as pd
import yaml
from tqdm import tqdm

from src.data.ssi_client import SSIDataClient
from src.data.fetch_daily import update_symbol_history
from src.data.loaders import read_parquet, save_parquet
from src.screening.light_market_scan import (
    latest_light_scan_snapshot,
    prepare_light_market_scan_features,
    save_auto_watchlist_outputs,
    select_auto_watchlist,
)


In [5]:
# Cell: Configure Fetch Mode And Date Range
with open("config/strategy_config.yaml", "r", encoding="utf-8") as f:
    strategy_config = yaml.safe_load(f)
universe = strategy_config.get("universe", {})

fetch_mode = "full_market_light"  # "watchlist" hoac "full_market_light"
refresh_securities_universe = True
max_symbols = None  # vi du: 200 de pilot truoc khi quet full market
watchlist_start_date = "2020-01-01"
full_market_lookback_days = 420
end_date = date.today().strftime("%Y-%m-%d")
deep_history_start_date = "2023-01-01"

phase1_candidate_pool_n = 60
phase1_top_liquid_universe_n = 100
phase2_final_watchlist_n = 10
phase1_sector_cap = 6
phase2_sector_cap = 2
phase2_deep_refresh_top_n = 60
phase_warning_threshold = 5
min_phase2_history_rows = 500
phase1_refresh_recent_days = 45
chunk_days = 29
request_sleep = 1.2

# Phase 1 universe hygiene
exclude_name_keywords = [
    "CHUNG QUYEN",
    "CW",
    "TRAI PHIEU",
    "BOND",
    "FUTURES",
    "HOP DONG TUONG LAI",
    "CHUNG CHI QUY",
    "ETF",
    "QUY DAU TU",
    "QUY DT",
]
keep_symbol_pattern = "letters_3"  # "alnum_1_3" an toan hon; "letters_1_3" rat gat va co the loai ca co phieu co so nhu C32, D2D, PC1
recent_liquidity_days = 7
min_recent_active_days = 3
min_recent_avg_value = 5_000_000_000.0
min_recent_median_value = 3_000_000_000.0

markets = universe.get("markets", [])
watchlist = universe.get("watchlist", [])

if fetch_mode == "watchlist":
    start_date = watchlist_start_date
else:
    start_date = (date.today() - timedelta(days=int(full_market_lookback_days))).strftime("%Y-%m-%d")

print("fetch_mode:", fetch_mode)
print("refresh_securities_universe:", refresh_securities_universe)
print("markets:", markets)
print("watchlist_size:", len(watchlist))
print("start_date:", start_date)
print("end_date:", end_date)
print("deep_history_start_date:", deep_history_start_date)
print("phase1_candidate_pool_n:", phase1_candidate_pool_n)
print("phase1_top_liquid_universe_n:", phase1_top_liquid_universe_n)
print("phase2_final_watchlist_n:", phase2_final_watchlist_n)
print("phase2_deep_refresh_top_n:", phase2_deep_refresh_top_n)
print("min_phase2_history_rows:", min_phase2_history_rows)
print("phase1_refresh_recent_days:", phase1_refresh_recent_days)
print("keep_symbol_pattern:", keep_symbol_pattern)
print("recent_liquidity_days:", recent_liquidity_days)
print("min_recent_avg_value:", min_recent_avg_value)


fetch_mode: full_market_light
refresh_securities_universe: True
markets: ['HOSE', 'HNX', 'UPCOM']
watchlist_size: 30
start_date: 2025-05-06
end_date: 2026-06-30
deep_history_start_date: 2023-01-01
phase1_candidate_pool_n: 60
phase1_top_liquid_universe_n: 100
phase2_final_watchlist_n: 10
phase2_deep_refresh_top_n: 60
min_phase2_history_rows: 500
phase1_refresh_recent_days: 45
keep_symbol_pattern: letters_3
recent_liquidity_days: 7
min_recent_avg_value: 5000000000.0


In [6]:
# Cell: Build Fetch Universe And Estimate Runtime
client = SSIDataClient(validate_credentials=True)
securities_dir = Path("data/raw/securities")
securities_dir.mkdir(parents=True, exist_ok=True)


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text.upper().strip()


def is_warrant_like_symbol(symbol: str) -> bool:
    symbol = str(symbol).upper().strip()
    return bool(re.fullmatch(r"C[A-Z]{2,5}\d{4}", symbol))


def is_allowed_cash_equity_symbol(symbol: str, pattern_mode: str = "letters_3") -> bool:
    symbol = str(symbol).upper().strip()
    if pattern_mode == "letters_3":
        return bool(re.fullmatch(r"[A-Z]{3}", symbol))
    if pattern_mode == "letters_1_3":
        return bool(re.fullmatch(r"[A-Z]{1,3}", symbol))
    return bool(re.fullmatch(r"[A-Z0-9]{1,3}", symbol))


if fetch_mode == "full_market_light":
    sec_frames = []
    for market in markets:
        path = securities_dir / f"{str(market).upper()}.parquet"
        if path.exists():
            sec = read_parquet(path)
            if not sec.empty:
                sec = sec.copy()
                sec["source_market"] = str(market).upper()
                sec_frames.append(sec)

    if not sec_frames:
        print("Khong tim thay securities cache local. Bat dau tai tu SSI...")
        refresh_securities_universe = True

    if refresh_securities_universe:
        sec_frames = []
        for market in markets:
            securities_df = client.get_securities(market)
            save_parquet(securities_df, securities_dir / f"{str(market).upper()}.parquet")
            print("saved securities:", market, securities_df.shape)
            if not securities_df.empty:
                securities_df = securities_df.copy()
                securities_df["source_market"] = str(market).upper()
                sec_frames.append(securities_df)

    if not sec_frames:
        raise FileNotFoundError(
            "Khong the build full market universe. Kiem tra SSI credentials, quota, network, hoac goi script update_securities.py truoc."
        )

    securities_master = pd.concat(sec_frames, ignore_index=True)
    symbol_col = "symbol" if "symbol" in securities_master.columns else "ticker"
    if symbol_col not in securities_master.columns:
        raise KeyError("Khong tim thay cot symbol/ticker trong securities universe")
    securities_master = securities_master.rename(columns={symbol_col: "symbol"}).copy()
    securities_master["symbol"] = securities_master["symbol"].astype(str).str.upper()

    name_cols = [col for col in ["stockname", "stockenname"] if col in securities_master.columns]
    if name_cols:
        instrument_text = securities_master[name_cols].fillna("").astype(str).agg(" ".join, axis=1).map(normalize_text)
        keyword_mask = instrument_text.apply(lambda text: any(keyword in text for keyword in exclude_name_keywords))
    else:
        keyword_mask = pd.Series(False, index=securities_master.index)

    warrant_symbol_mask = securities_master["symbol"].map(is_warrant_like_symbol)
    symbol_pattern_mask = securities_master["symbol"].map(lambda s: is_allowed_cash_equity_symbol(s, keep_symbol_pattern))
    exclusion_mask = keyword_mask | warrant_symbol_mask | (~symbol_pattern_mask)

    excluded_instruments = securities_master[exclusion_mask].copy()
    securities_master = securities_master[~exclusion_mask].copy()
    raw_fetch_symbols = sorted(securities_master["symbol"].dropna().unique().tolist())
    # Dynamic pre-filter using latest market daily data to skip illiquid stocks
    print("Performing dynamic liquidity pre-filtering...")
    five_days_ago = (date.today() - timedelta(days=7)).strftime("%Y-%m-%d")
    symbol_value_sum = {}
    symbol_value_count = {}
    for m in markets:
        try:
            print(f"Fetching bulk market data for {m} to filter liquidity...")
            m_df = client.get_daily_stock_price('', five_days_ago, end_date, market=m)
            if not m_df.empty:
                m_df["totaltradedvalue"] = pd.to_numeric(m_df["totaltradedvalue"], errors="coerce").fillna(0.0)
                m_df["symbol"] = m_df["symbol"].astype(str).str.upper().str.strip()
                for _, row in m_df.iterrows():
                    sym = row["symbol"]
                    val = row["totaltradedvalue"]
                    if len(sym) == 3:
                        symbol_value_sum[sym] = symbol_value_sum.get(sym, 0.0) + val
                        symbol_value_count[sym] = symbol_value_count.get(sym, 0) + 1
        except Exception as e:
            print(f"Warning: Could not fetch bulk market data for {m}: {e}")
            
    liquid_symbols = set()
    for sym, val_sum in symbol_value_sum.items():
        cnt = symbol_value_count.get(sym, 1)
        if (val_sum / cnt) >= 1500000000.0:
            liquid_symbols.add(sym)
            
    fetch_symbols = []
    skipped_illiquid = 0
    for symbol in raw_fetch_symbols:
        if len(liquid_symbols) > 0:
            if symbol not in liquid_symbols:
                skipped_illiquid += 1
                continue
        else:
            path = Path(f"data/raw/daily_ohlcv/{symbol}.parquet")
            if path.exists():
                try:
                    df = read_parquet(path)
                    if not df.empty and df.tail(20)["value"].mean() < 1500000000.0:
                        skipped_illiquid += 1
                        continue
                except Exception:
                    pass
        fetch_symbols.append(symbol)
    print(f"Skipped {skipped_illiquid} illiquid symbols (recent avg value < 1.5B VND). Remaining: {len(fetch_symbols)} symbols.")
else:
    excluded_instruments = pd.DataFrame()
    securities_master = pd.DataFrame()
    fetch_symbols = sorted({str(symbol).upper() for symbol in watchlist})

if max_symbols is not None:
    fetch_symbols = fetch_symbols[: int(max_symbols)]

light_days = max(1, (pd.to_datetime(end_date) - pd.to_datetime(start_date)).days + 1)
deep_days = max(1, (pd.to_datetime(end_date) - pd.to_datetime(deep_history_start_date)).days + 1)
light_requests_per_symbol = int((light_days + chunk_days - 1) // chunk_days)
deep_requests_per_symbol = int((deep_days + chunk_days - 1) // chunk_days)
phase1_seconds_est = len(fetch_symbols) * light_requests_per_symbol * request_sleep
phase2_seconds_est = min(len(fetch_symbols), int(phase2_deep_refresh_top_n)) * deep_requests_per_symbol * request_sleep if fetch_mode == "full_market_light" else 0.0

universe_summary = {
    "fetch_mode": fetch_mode,
    "markets_requested": ", ".join(markets),
    "symbols_to_fetch": len(fetch_symbols),
    "excluded_by_metadata_or_symbol": int(len(excluded_instruments)),
    "start_date": start_date,
    "end_date": end_date,
    "deep_history_start_date": deep_history_start_date,
    "phase1_candidate_pool_n": phase1_candidate_pool_n,
    "phase2_final_watchlist_n": phase2_final_watchlist_n,
    "phase2_deep_refresh_top_n": phase2_deep_refresh_top_n,
    "light_requests_per_symbol": light_requests_per_symbol,
    "deep_requests_per_symbol": deep_requests_per_symbol,
    "phase1_est_hours": round(phase1_seconds_est / 3600, 2),
    "phase2_est_hours": round(phase2_seconds_est / 3600, 2),
    "total_est_hours": round((phase1_seconds_est + phase2_seconds_est) / 3600, 2),
    "max_symbols": max_symbols,
}
display(pd.DataFrame([universe_summary]))
if not excluded_instruments.empty:
    display(excluded_instruments.head(30))
fetch_symbols[:20]


saved securities: HOSE (736, 4)
saved securities: HNX (380, 4)
saved securities: UPCOM (887, 4)
Performing dynamic liquidity pre-filtering...
Fetching bulk market data for HOSE to filter liquidity...
Fetching bulk market data for HNX to filter liquidity...
Fetching bulk market data for UPCOM to filter liquidity...
Skipped 1245 illiquid symbols (recent avg value < 1.5B VND). Remaining: 210 symbols.


,fetch_mode,markets_requested,symbols_to_fetch,excluded_by_metadata_or_symbol,start_date,end_date,deep_history_start_date,phase1_candidate_pool_n,phase2_final_watchlist_n,phase2_deep_refresh_top_n,light_requests_per_symbol,deep_requests_per_symbol,phase1_est_hours,phase2_est_hours,total_est_hours,max_symbols
0,full_market_light,"HOSE, HNX, UPCOM",210,548,2025-05-06,2026-06-30,2023-01-01,60,10,60,15,45,1.05,0.9,1.95,None


,market,symbol,stockname,stockenname,source_market
43,HOSE,C32,CTCP CIC39,CIC39 Corporation,HOSE
44,HOSE,C47,CTCP XAY DUNG 47,Construction Joint Stock Company 47,HOSE
45,HOSE,CACB2502,Chứng quyền ACB/15M/SSI/C/EU/Cash-18,None,HOSE
46,HOSE,CACB2510,Chứng quyền ACB/12M/SSI/C/EU/Cash-20,None,HOSE
47,HOSE,CACB2511,Chứng quyền ACB/15M/SSI/C/EU/Cash-20,None,HOSE
48,HOSE,CACB2512,Chứng quyền ACB/MBS/C/EU/CASH/6M/25-01,None,HOSE
49,HOSE,CACB2513,Chứng quyền.ACB.VND.M.CA.T.2025.1,None,HOSE
50,HOSE,CACB2514,Chứng quyền.ACB.VND.M.CA.T.2025.2,None,HOSE
51,HOSE,CACB2515,Chứng quyền.ACB.VND.M.CA.T.2025.3,None,HOSE
52,HOSE,CACB2516,Chứng quyền ACB/12M/KAFI/C/EU/CASH/2501,None,HOSE


['AAA',
 'AAS',
 'AAV',
 'ABB',
 'ABW',
 'ACB',
 'ACV',
 'AGG',
 'AGR',
 'AIG',
 'ANV',
 'APF',
 'APG',
 'ASM',
 'ASP',
 'BAF',
 'BCM',
 'BDT',
 'BFC',
 'BID']

In [7]:
# Cell: Fetch One Symbol Smoke Test
sample_symbol = fetch_symbols[0]
sample_df = update_symbol_history(
    client=client,
    symbol=sample_symbol,
    start_date=start_date,
    end_date=end_date,
    output_dir="data/raw/daily_ohlcv",
    chunk_days=chunk_days,
    request_sleep=request_sleep,
    force_refresh=False,
    refresh_recent_days=phase1_refresh_recent_days,
)

print("sample_symbol:", sample_symbol)
sample_df.tail(), sample_df.shape


[OK] AAA: 16/05/2026 -> 14/06/2026: 20 rows
[OK] AAA: 15/06/2026 -> 30/06/2026: 12 rows
sample_symbol: AAA


(     symbol market       date  open  high   low  close   volume         value
 1613    AAA   HOSE 2026-06-24  7390  7440  7320   7400   907200  6.693778e+09
 1614    AAA   HOSE 2026-06-25  7400  7500  7390   7430  1695700  1.259950e+10
 1615    AAA   HOSE 2026-06-26  7430  7450  7340   7370   627900  4.640249e+09
 1616    AAA   HOSE 2026-06-29  7330  7450  7290   7330   861600  6.332768e+09
 1617    AAA   HOSE 2026-06-30  7330  7370  7310   7350    38300  2.816230e+08,
 (1618, 9))

In [ ]:
# Cell: Bulk Fetch Selected Universe
all_frames = []
coverage_rows = []

# Fetch reference symbol (like FPT) to get the latest trading date in the market
print("Checking latest trading date from reference symbol FPT...")
ref_df = update_symbol_history(
    client=client,
    symbol="FPT",
    start_date=start_date,
    end_date=end_date,
    output_dir="data/raw/daily_ohlcv",
    chunk_days=chunk_days,
    request_sleep=request_sleep,
    force_refresh=False,
    refresh_recent_days=phase1_refresh_recent_days,
)
market_latest_date = ref_df["date"].max() if not ref_df.empty else None
print(f"Latest market trading date: {market_latest_date}")

for symbol in tqdm(fetch_symbols):
    df = update_symbol_history(
        client=client,
        symbol=symbol,
        start_date=start_date,
        end_date=end_date,
        output_dir="data/raw/daily_ohlcv",
        chunk_days=chunk_days,
        request_sleep=request_sleep,
        force_refresh=False,
        refresh_recent_days=phase1_refresh_recent_days,
        market_latest_date=market_latest_date,
    )
    if not df.empty:
        all_frames.append(df)
        coverage_rows.append({
            "symbol": symbol,
            "start_date": df["date"].min(),
            "end_date": df["date"].max(),
            "rows": len(df),
        })

if not all_frames:
    raise ValueError("Không fetch được bất kỳ dữ liệu nào.")

daily = pd.concat(all_frames, ignore_index=True)

# GUARDED WRITE (docs/reports/codebase_audit/NOTEBOOK_AUDIT_FULL.md NV1) --
# this cell writes straight from a raw fetch (no standardize_ohlcv/validate_ohlcv
# applied) into the official clean_daily_ohlcv.parquet, read by ~20 other
# scripts/modules. Same guard pattern as 02_market_research/05+06.
from src.data.production_guard import guard_production_write, ProductionWriteBlocked

output_path = "data/processed/clean_daily_ohlcv.parquet"
try:
    guard_production_write(output_path)
    save_parquet(daily, output_path)
    print(f"Da ghi {output_path} (ALLOW_PRODUCTION_WRITE=1). Shape:", daily.shape)
except ProductionWriteBlocked as e:
    print(f"KHONG ghi {output_path} -- {e}")
    print("Fetched shape (not saved):", daily.shape)
